Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [2]:
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [3]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                               Gene    Sequence                      Glycan                             
ENSP00000315945@615;ENSP00000393474@625            CD163L1 AAAVVCSQLDCPSSIIGMGLGNASTGYGK N5H6F3S0G0             13.202015   
ENSP00000349437@2122                               IGF2R   AACAVKPQEVQMVNGTITNPINGK      N2H12F0S0G0            13.284213   
                                                                                         N4H5F1S2G0             14.235520   
ENSP00000509841@48;ENSP00000273353@375             MYH15   AAFLMGINSSELVK                N3H5F0S2G0             15.191432   
ENSP00000296504@78                                 SAP30   AAGNASFSK                     N3H5F1S2G0             13.867617   
...                                                                                                                   ...   
ENSP00000261590@458                                DSG2    YVQNGTYTVK                    N4H5F4S1G0             11.978667   
                                                                                         N4H6F1S1G0             13.358014   
                                                                                         N5H6F3S1G0             11.445073   
ENSP00000363397@195                                UGCG    YYISANVTGFKCVTGMSCLMRK        N11H9F1S0G0            11.704895   
ENSP00000413130@124;ENSP00000438497@212;ENSP000... GNS     YYNYTLSINGK                   N4H5F3S1G0             12.903119   

                                                                                                        pool_01  \
Site                                               Gene    Sequence                      Glycan                   
ENSP00000315945@615;ENSP00000393474@625            CD163L1 AAAVVCSQLDCPSSIIGMGLGNASTGYGK N5H6F3S0G0   13.202015   
ENSP00000349437@2122                               IGF2R   AACAVKPQEVQMVNGTITNPINGK      N2H12F0S0G0  13.284213   
                                                                                         N4H5F1S2G0   14.235520   
ENSP00000509841@48;ENSP00000273353@375             MYH15   AAFLMGINSSELVK                N3H5F0S2G0   15.191432   
ENSP00000296504@78                                 SAP30   AAGNASFSK                     N3H5F1S2G0   13.867617   
...                                                                                                         ...   
ENSP00000261590@458                                DSG2    YVQNGTYTVK                    N4H5F4S1G0         NaN   
                                                                                         N4H6F1S1G0         NaN   
                                                                                         N5H6F3S1G0         NaN   
ENSP00000363397@195                                UGCG    YYISANVTGFKCVTGMSCLMRK        N11H9F1S0G0        NaN   
ENSP00000413130@124;ENSP00000438497@212;ENSP000... GNS     YYNYTLSINGK                   N4H5F3S1G0         NaN   

                                                                                                       QC1_C_01  \
Site                                               Gene    Sequence                      Glycan                   
ENSP00000315945@615;ENSP00000393474@625            CD163L1 AAAVVCSQLDCPSSIIGMGLGNASTGYGK N5H6F3S0G0   12.831488   
ENSP00000349437@2122                               IGF2R   AACAVKPQEVQMVNGTITNPINGK      N2H12F0S0G0  16.038965   
                                                                                         N4H5F1S2G0   16.501786   
ENSP00000509841@48;ENSP00000273353@375             MYH15   AAFLMGINSSELVK                N3H5F0S2G0   14.647075   
ENSP00000296504@78                                 SAP30   AAGNASFSK                     N3H5F1S2G0   13.402210   
...                                                                                                         ...   
ENSP00000261590@458                                DSG2    Y

In [4]:
meta_path= os.path.join(meta_dir, "PDAC_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,Tobacco_smoking_history,KRAS_mutation,TP53_mutation,SMAD4_mutation,CDKN2A_mutation
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,ORD,BIN,BIN,BIN,BIN
0,C3L-00017,69,Male,4.5,G2 Moderately differentiated,Not identified,pT3,pN0,Stage II,28.36,past smoker,1,0,0,0
1,C3L-00102,42,Male,3.0,G3 Poorly differentiated,Not identified,pT3,pN1,Stage II,26.93,non-smoker,1,1,0,1
2,C3L-00277,69,Male,5.0,G2 Moderately differentiated,Not identified,pT3,pN1,Stage II,24.00,non-smoker,1,0,1,0
3,C3L-00589,80,Male,2.5,G2 Moderately differentiated,Not identified,pT3,pN1,Stage II,27.43,past smoker,1,1,0,0
4,C3L-00598,61,Female,3.0,G2 Moderately differentiated,Present,pT3,pN1,Stage II,16.00,current smoker,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,C3N-03884,64,Female,2.3,G2 Moderately differentiated,Not identified,pT2,pN2,Stage III,20.40,non-smoker,1,1,0,0
101,C3N-04119,54,Male,3.0,G2 Moderately differentiated,Not identified,pT2,pN1,Stage II,23.66,non-smoker,1,1,1,1
102,C3N-04126,42,Male,3.5,G3 Poorly differentiated,Not identified,pT2,pN2,Stage III,24.38,non-smoker,1,1,0,0


In [5]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,C3L-00017,Male,Stage II
1,C3L-00102,Male,Stage II
2,C3L-00277,Male,Stage II
3,C3L-00589,Male,Stage II
4,C3L-00598,Female,Stage II
...,...,...,...
100,C3N-03884,Female,Stage III
101,C3N-04119,Male,Stage II
102,C3N-04126,Male,Stage III
103,C3N-04282,Male,Stage IV


In [6]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,C3L-00017,Male,Stage II
1,C3L-00102,Male,Stage II
2,C3L-00277,Male,Stage II
3,C3L-00589,Male,Stage II
4,C3L-00598,Female,Stage II
5,C3L-00599,Male,Stage II
6,C3L-00622,Male,Stage II
7,C3L-00625,Female,Stage II
8,C3L-00819,Male,Stage II
9,C3L-00928,Female,Stage II


In [7]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

105

In [8]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [9]:
meta3.head(17)

,Sample.ID,Sex,Stage
100,C3N-03884_T_01,Female,Stage III
37,C3L-03123_T_01,Female,Stage II
19,C3L-01687_T_01,Female,Stage II
3,C3L-00589_T_01,Male,Stage II
5,C3L-00599_T_01,Male,Stage II
38,C3L-03356_T_02,Male,Stage II
41,C3L-03630_T_02,Male,Stage II
34,C3L-02890_T_02,Male,Stage II
102,C3N-04126_T_03,Male,Stage III
30,C3L-02610_T_03,Male,Stage II


In [10]:
meta3 = meta3.replace(np.nan,'NA')

In [11]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [12]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [13]:
data_df.head(2)

,,,,Intensity.Reference,pool_01,QC1_C_01,C3N-03884_T_01,C3L-00589_N_F_01,C3L-03123_N_01,C3L-01687_N_F_01,C3L-03123_T_01,C3L-01687_T_01,C3L-00589_T_01,...,C3L-03513_N_25,"WU-PDA1, Tumor_25",C3L-03514_N_25,C3L-03515_N_25,ref_25,C3L-07032_N_25,C3L-07033_N_25,C3L-07034_N_25,C3L-07035_N_25,C3L-07036_N_25
Site,Gene,Sequence,Glycan,,,,,,,,,,,,,,,,,,,,,
ENSP00000315945@615;ENSP00000393474@625,CD163L1,AAAVVCSQLDCPSSIIGMGLGNASTGYGK,N5H6F3S0G0,13.202015,13.202015,12.831488,13.865213,15.288988,13.006694,14.418222,12.672634,13.835176,13.411525,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSP00000349437@2122,IGF2R,AACAVKPQEVQMVNGTITNPINGK,N2H12F0S0G0,13.284213,13.284213,16.038965,12.823154,11.902429,13.240259,12.730439,14.161635,13.242722,13.070941,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
samples = meta3['Sample.ID'].to_list()

In [15]:
df2 = data_df.loc[:,samples].dropna()

In [16]:
df2.shape

(801, 105)

In [17]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [18]:
data2.shape

(771, 105)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [19]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [20]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [21]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N4H...,F+S
1,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
2,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
3,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
4,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,F+S
...,...,...
766,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,only_S
767,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
768,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
769,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S


Map glycan types with colors.

In [22]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [23]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [24]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)